# Solar Filament Segmentation Challenge 2026 — Final Kaggle Pipeline

**Goal:** a real, reproducible model-generated submission with held-out evaluation and instance-aware post-processing.

This notebook intentionally does **not** install or downgrade PyTorch/Torchvision. The earlier notebook logged validation Dice near zero and produced test masks from a hard-coded payload; those predictions are not used here. Every submitted mask in this notebook is created by inference on the attached test images.

Official metric guidance: [competition overview](https://www.kaggle.com/competitions/filament-segmentation-2026/overview/final-submission) · [organizer self-evaluation notebook](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook).

**First run:** set Kaggle Accelerator to GPU. This notebook's `RUN_MODE = "competition"` does a smoke update, trains a baseline, tunes post-processing, measures a separate validation holdout, retrains on all available labelled annotation views, and writes `/kaggle/working/submission.csv`. Switch to `"smoke"` only to verify the environment quickly; smoke-mode output is not intended as a competition entry.

In [ ]:
# ============================================================
# CELL 1 — GITHUB SOURCE BOOTSTRAP
# ============================================================
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = "https://github.com/ak11pro/Chatgpt.web.git"
REPO_BRANCH = "codex/audit-provided-notebook-for-weaknesses"
REPO_DIR = Path("/kaggle/working/Chatgpt.web")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)

PIPELINE_PATH = REPO_DIR / "solar_filament" / "pipeline.py"
assert PIPELINE_PATH.is_file(), f"Pipeline source missing: {PIPELINE_PATH}"
sys.path.insert(0, str(REPO_DIR))

REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

print("Repository:", REPO_DIR)
print("Branch:", REPO_BRANCH)
print("Commit:", REPO_COMMIT)
print("Pipeline:", PIPELINE_PATH)


In [ ]:
# ============================================================
# CELL 2 — INSTALL ONLY MISSING NON-TORCH DEPENDENCIES
# Do not upgrade/downgrade torch, torchvision, or numpy here.
# ============================================================
import importlib.util

PACKAGE_FOR_MODULE = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "cv2": "opencv-python-headless",
    "scipy": "scipy",
    "skimage": "scikit-image",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "pycocotools": "pycocotools",
}

missing_packages = [
    package for module, package in PACKAGE_FOR_MODULE.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    print("Installing missing packages only:", missing_packages)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_packages],
        check=True,
    )
else:
    print("All non-Torch dependencies already exist.")


In [ ]:
# ============================================================
# CELL 3 — SAFE TORCH/CUDA PREFLIGHT
# ============================================================
try:
    import torch
except Exception as exc:
    print("PyTorch itself failed to import.")
    print(type(exc).__name__, str(exc))
    subprocess.run(
        [sys.executable, "-m", "pip", "show", "torch", "torchvision"],
        check=False,
    )
    raise RuntimeError(
        "Do not reinstall a random Torch wheel. Restart the Kaggle session, "
        "remove any earlier torch/torchvision pip-install cell, and run this "
        "notebook in a fresh Kaggle GPU session."
    ) from exc

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

TORCHVISION_OK = False
TORCHVISION_ERROR = None
try:
    import torchvision
    from torchvision.models import ResNet34_Weights, resnet34
    TORCHVISION_OK = True
    print("Torchvision:", torchvision.__version__)
    print("Torchvision ResNet-34: available")
except Exception as exc:
    TORCHVISION_ERROR = f"{type(exc).__name__}: {exc}"
    print("Torchvision import failed:", TORCHVISION_ERROR)
    print("The pipeline has a pure-PyTorch ResNet-shaped fallback. It will not have ImageNet weights.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: CUDA is unavailable. Competition mode requires a GPU.")

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# CELL 4 — IMPORT PIPELINE + RUN SETTINGS
# ============================================================
import gc
import hashlib
import json
import logging
import random
from dataclasses import asdict
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import DataLoader

from solar_filament.pipeline import (
    PipelineConfig,
    FilamentSegmentationModel,
    SolarFilamentDataset,
    append_experiment,
    build_grad_scaler,
    build_instance_masks,
    build_semantic_mask,
    cache_validation_predictions,
    compute_pq,
    compound_loss,
    create_submission,
    environment_report,
    find_competition_data,
    group_train_validation_split,
    load_coco_annotations,
    measure_foreground_prevalence,
    optimize_postprocessing,
    run_synthetic_metric_tests,
    set_seed,
    train_one_epoch,
    validate,
    reconstruct_instances,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Use "smoke" for a quick compatibility check; use "competition" for the full run.
RUN_MODE = "competition"
FINAL_RETRAIN_ALL_DATA = True

SEED = 42
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_TAG += "_" + REPO_COMMIT[:7]

if RUN_MODE not in {"smoke", "competition"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'competition'.")
if RUN_MODE == "competition" and not torch.cuda.is_available():
    raise RuntimeError("Set Kaggle Accelerator = GPU before running competition mode.")

config = PipelineConfig(
    seed=SEED,
    image_size=2048,
    train_tile_size=512 if RUN_MODE == "smoke" else 768,
    tile_overlap=0.35,
    inference_batch_size=1 if RUN_MODE == "smoke" else 2,
    in_channels=1,
    pretrained_encoder=True,
    epochs=1 if RUN_MODE == "smoke" else 18,
    batch_size=1,
    gradient_accumulation=2,
    learning_rate=2e-4,
    weight_decay=1e-4,
    num_workers=2,
    use_amp=True,
    val_fraction=0.20,
    threshold=0.40,
    min_component_area=200,
    watershed=True,
    watershed_peak_distance=18,
    boundary_mode="guidance",
    boundary_head=True,
    use_tta=False,
    use_solar_disk_mask=True,  # tested both on/off by PQ-driven post-processing search
    max_train_images=None,
    samples_per_epoch=32 if RUN_MODE == "smoke" else 900,
    mask_cache_dir="/kaggle/working/solar_filament_mask_cache_v2",
)

set_seed(SEED)
torch.set_num_threads(max(1, min(8, os.cpu_count() or 2)))

print(json.dumps(environment_report(config), indent=2))
print("Run mode:", RUN_MODE)
print("Run tag:", RUN_TAG)
run_synthetic_metric_tests()
print("Synthetic PQ tests passed.")


In [ ]:
# ============================================================
# CELL 5 — FIND DATA + SPLIT BY PHYSICAL OBSERVATION
# Every annotation view of the same JPEG stays in one split.
# ============================================================
data = find_competition_data("/kaggle/input")
index = load_coco_annotations(data.annotation_json)

train_ids, val_ids = group_train_validation_split(
    index, config.val_fraction, config.seed
)

physical_train = {
    Path(index.images[i]["file_name"]).name for i in train_ids
}
physical_val = {
    Path(index.images[i]["file_name"]).name for i in val_ids
}
assert physical_train.isdisjoint(physical_val), "Physical observation leakage detected."

val_stems = sorted(physical_val)
rng = np.random.default_rng(SEED)
val_stems_array = np.asarray(val_stems, dtype=object)
rng.shuffle(val_stems_array)

cut = max(1, len(val_stems_array) // 2)
tune_pool = list(val_stems_array[:cut])
holdout_stems = set(val_stems_array[cut:].tolist())

# Limit expensive post-processing sweeps to a reproducible subset of tuning
# observations. The holdout remains untouched for the final reported score.
n_tune = min(32 if RUN_MODE == "competition" else 4, len(tune_pool))
tune_stems = set(rng.choice(tune_pool, size=n_tune, replace=False).tolist())

tune_ids = [
    i for i in val_ids
    if Path(index.images[i]["file_name"]).name in tune_stems
]
holdout_ids = [
    i for i in val_ids
    if Path(index.images[i]["file_name"]).name in holdout_stems
]

quick_stems = set(rng.choice(
    sorted(tune_stems),
    size=min(12 if RUN_MODE == "competition" else 2, len(tune_stems)),
    replace=False,
).tolist())
quick_val_ids = [
    i for i in tune_ids
    if Path(index.images[i]["file_name"]).name in quick_stems
]

assert tune_ids and holdout_ids and quick_val_ids
assert {Path(index.images[i]["file_name"]).name for i in tune_ids}.isdisjoint(
    {Path(index.images[i]["file_name"]).name for i in holdout_ids}
)

split_summary = {
    "train_annotation_entries": len(train_ids),
    "validation_annotation_entries": len(val_ids),
    "training_physical_images": len(physical_train),
    "validation_physical_images": len(physical_val),
    "postprocess_tuning_physical_images": len(tune_stems),
    "final_holdout_physical_images": len(holdout_stems),
    "quick_validation_physical_images": len(quick_stems),
    "train_instance_count": sum(len(index.annotations_by_image.get(i, [])) for i in train_ids),
}
print(json.dumps(split_summary, indent=2))


In [ ]:
# ============================================================
# CELL 6 — VISUAL ANNOTATION SANITY CHECK
# Inspect this grid. Stop if polygons are empty or misaligned.
# ============================================================
vis_ids = rng.choice(
    train_ids,
    size=min(12, len(train_ids)),
    replace=False,
)
fig, axes = plt.subplots(3, 4, figsize=(18, 13))

for ax, image_id in zip(axes.flat, vis_ids):
    meta = index.images[image_id]
    img_path = data.train_images / meta["file_name"]
    with Image.open(img_path) as handle:
        image = np.asarray(handle.convert("L"))
    instances = build_instance_masks(index, image_id)
    semantic = build_semantic_mask(index, image_id)

    assert image.shape == semantic.shape == (2048, 2048)
    assert all(m.shape == image.shape and m.any() for m in instances)

    ax.imshow(image, cmap="gray")
    ax.imshow(semantic, cmap="Reds", alpha=0.40)
    ax.set_title(f"{Path(meta['file_name']).name}\nviews={len(instances)}")
    ax.axis("off")

for ax in axes.flat[len(vis_ids):]:
    ax.axis("off")

fig.tight_layout()
sanity_path = WORKING / f"annotation_sanity_{RUN_TAG}.png"
fig.savefig(sanity_path, dpi=130)
plt.show()
print("Saved annotation sanity grid:", sanity_path)


In [ ]:
# ============================================================
# CELL 7 — BUILD CACHED DATASETS + QUICK VALIDATION SUBSET
# Masks are rasterized once into a compressed, versioned cache.
# ============================================================
print("Building training dataset and crop candidates...")
train_dataset = SolarFilamentDataset(
    index, data.train_images, train_ids, config, training=True
)
quick_val_dataset = SolarFilamentDataset(
    index, data.train_images, quick_val_ids, config, training=False
)
tune_dataset = SolarFilamentDataset(
    index, data.train_images, tune_ids, config, training=False
)
holdout_dataset = SolarFilamentDataset(
    index, data.train_images, holdout_ids, config, training=False
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(config.num_workers > 0),
)

print("Device:", device)
print("Training crop samples per epoch:", len(train_dataset))
print("Quick validation views:", len(quick_val_dataset))
print("Tuning views:", len(tune_dataset))
print("Independent holdout views:", len(holdout_dataset))


In [ ]:
# ============================================================
# CELL 8 — GPU FORWARD/BACKWARD SMOKE TEST
# Catch import, shape, loss, and CUDA issues before long training.
# ============================================================
model = FilamentSegmentationModel(config).to(device)
print("ImageNet-pretrained encoder loaded:", getattr(model, "pretrained_loaded", False))

batch = next(iter(train_loader))
images = batch["image"].to(device)
semantic = batch["semantic"].to(device)
boundary = batch["boundary"].to(device)

model.train()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

optimizer.zero_grad(set_to_none=True)
outputs = model(images)
loss = compound_loss(
    outputs,
    {"semantic": semantic, "boundary": boundary},
    config,
    pos_weight=20.0,
)
assert torch.isfinite(loss).item(), f"Non-finite smoke-test loss: {loss.item()}"
loss.backward()
optimizer.step()

print({
    "input_shape": tuple(images.shape),
    "semantic_output_shape": tuple(outputs["semantic"].shape),
    "boundary_output_shape": tuple(outputs["boundary"].shape),
    "smoke_loss": float(loss.detach().cpu()),
    "finite_gradients": all(
        p.grad is None or torch.isfinite(p.grad).all().item()
        for p in model.parameters()
    ),
})
del batch, images, semantic, boundary, outputs
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# CELL 9 — TRAIN AND SELECT CHECKPOINT ON QUICK HELD-OUT PQ
# ============================================================
# This prevalence sample is intentionally bounded to keep startup time sane.
prevalence = measure_foreground_prevalence(
    index, train_ids[:min(48, len(train_ids))]
)
pos_weight = float(np.clip(prevalence["background_to_foreground"], 1.0, 50.0))
print("Sampled prevalence:", prevalence)
print("BCE pos_weight:", pos_weight)

# Reinitialize after the smoke update so the measured run begins cleanly.
model = FilamentSegmentationModel(config).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config.epochs
)
scaler = build_grad_scaler(device, config.use_amp)

best_path = WORKING / f"best_model_{RUN_TAG}.pth"
history = []
best_metric = -1.0
best_epoch = 0
epochs_without_improvement = 0
patience = 5
started = time.perf_counter()

for epoch in range(config.epochs):
    epoch_started = time.perf_counter()

    train_loss = train_one_epoch(
        model, train_loader, optimizer, scaler,
        config, device, pos_weight
    )
    scheduler.step()

    quick_metrics = validate(
        model, quick_val_dataset, config, device
    )
    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "epoch_seconds": time.perf_counter() - epoch_started,
        **quick_metrics,
    }
    history.append(row)
    print(json.dumps(row, indent=2, default=float))

    if quick_metrics["global_pq"] > best_metric + 1e-6:
        best_metric = quick_metrics["global_pq"]
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        torch.save({
            "model": model.state_dict(),
            "config": asdict(config),
            "metrics": quick_metrics,
            "epoch": best_epoch,
            "pos_weight": pos_weight,
            "repo_commit": REPO_COMMIT,
        }, best_path)
        print("Saved improved checkpoint:", best_path)
    else:
        epochs_without_improvement += 1

    if RUN_MODE == "competition" and epochs_without_improvement >= patience:
        print("Early stopping: quick validation PQ has not improved.")
        break

pd.DataFrame(history).to_csv(WORKING / f"training_history_{RUN_TAG}.csv", index=False)
if not best_path.exists():
    raise RuntimeError("No checkpoint was saved; refusing untrained inference.")

checkpoint = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model"])
print("Selected epoch:", best_epoch)
print("Selected quick global PQ:", best_metric)
print("Training seconds:", round(time.perf_counter() - started, 1))


In [ ]:
# ============================================================
# CELL 10 — TUNE POST-PROCESSING ON TUNING VIEWS ONLY
# The final holdout is not used by this search.
# ============================================================
tune_cache_dir = WORKING / f"tune_cache_{RUN_TAG}"
tune_cache = cache_validation_predictions(
    model, tune_dataset, config, device, tune_cache_dir
)

optimized_config, search_table = optimize_postprocessing(
    [entry[0] for entry in tune_cache],
    [entry[1] for entry in tune_cache],
    [entry[2] for entry in tune_cache],
    config,
    disk_masks=[entry[3] for entry in tune_cache],
)

search_path = WORKING / f"postprocessing_search_{RUN_TAG}.csv"
search_table.to_csv(search_path, index=False)

print("Best post-processing candidates:")
display(search_table.head(12))
print("Selected configuration:")
print(json.dumps(asdict(optimized_config), indent=2))


In [ ]:
# ============================================================
# CELL 11 — FINAL INDEPENDENT HELD-OUT EVALUATION
# Do not make further tuning decisions from these results.
# ============================================================
holdout_started = time.perf_counter()
holdout_metrics = validate(
    model,
    holdout_dataset,
    optimized_config,
    device,
)
holdout_seconds = time.perf_counter() - holdout_started

print("INDEPENDENT HELD-OUT METRICS (measured)")
print(json.dumps(holdout_metrics, indent=2))
print("Holdout evaluation seconds:", round(holdout_seconds, 1))

with (WORKING / f"holdout_metrics_{RUN_TAG}.json").open("w") as handle:
    json.dump(holdout_metrics, handle, indent=2)

with (WORKING / f"optimized_config_{RUN_TAG}.json").open("w") as handle:
    json.dump(asdict(optimized_config), handle, indent=2)

append_experiment(
    WORKING / "experiments.csv",
    f"resnet34_unet_boundary_{RUN_TAG}",
    optimized_config,
    holdout_metrics,
    time.perf_counter() - started,
    pos_weight,
    split_summary,
)


In [ ]:
# ============================================================
# CELL 12 — RETRAIN FINAL MODEL ON ALL LABELLED ANNOTATION VIEWS
# Hyperparameters were chosen before using the held-out metrics.
# ============================================================
if RUN_MODE == "competition" and FINAL_RETRAIN_ALL_DATA:
    # Free the validation model, dataloaders, and cached probability maps before
    # allocating the all-data model. This helps avoid GPU/RAM exhaustion.
    del model, optimizer, scheduler, scaler
    del train_loader, train_dataset, quick_val_dataset, tune_dataset, holdout_dataset
    del tune_cache
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    print("Building all-data training dataset...")
    all_train_ids = sorted(index.images)
    final_config = PipelineConfig(**{
        **asdict(optimized_config),
        "epochs": best_epoch,
        "samples_per_epoch": config.samples_per_epoch,
        "max_train_images": None,
    })

    final_dataset = SolarFilamentDataset(
        index, data.train_images, all_train_ids, final_config, training=True
    )
    final_loader = DataLoader(
        final_dataset,
        batch_size=final_config.batch_size,
        shuffle=True,
        num_workers=final_config.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(final_config.num_workers > 0),
    )

    final_model = FilamentSegmentationModel(final_config).to(device)
    final_optimizer = torch.optim.AdamW(
        final_model.parameters(),
        lr=final_config.learning_rate,
        weight_decay=final_config.weight_decay,
    )
    final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        final_optimizer, T_max=max(1, final_config.epochs)
    )
    final_scaler = build_grad_scaler(device, final_config.use_amp)
    final_training_history = []

    for epoch in range(final_config.epochs):
        epoch_started = time.perf_counter()
        loss = train_one_epoch(
            final_model,
            final_loader,
            final_optimizer,
            final_scaler,
            final_config,
            device,
            pos_weight,
        )
        final_scheduler.step()
        row = {
            "epoch": epoch + 1,
            "epochs": final_config.epochs,
            "train_loss": loss,
            "seconds": time.perf_counter() - epoch_started,
        }
        final_training_history.append(row)
        print(row)

    final_model_path = WORKING / f"final_model_all_data_{RUN_TAG}.pth"
    torch.save({
        "model": final_model.state_dict(),
        "config": asdict(final_config),
        "trained_on_all_annotation_views": True,
        "selected_epoch_count": best_epoch,
        "repo_commit": REPO_COMMIT,
    }, final_model_path)
    pd.DataFrame(final_training_history).to_csv(
        WORKING / f"final_training_history_{RUN_TAG}.csv",
        index=False,
    )
    model_for_submission = final_model
    submission_config = final_config
    del final_loader, final_dataset, final_optimizer, final_scheduler, final_scaler
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("Saved all-data checkpoint:", final_model_path)
else:
    model_for_submission = model
    submission_config = optimized_config
    final_model_path = best_path
    print("Using best validation-selected model; no all-data retrain requested.")


In [ ]:
# ============================================================
# CELL 13 — PREDICT ALL TEST IMAGES + VALIDATE SUBMISSION CSV
# ============================================================
if RUN_MODE == "competition":
    submission_path = WORKING / "submission.csv"

    submission = create_submission(
        model=model_for_submission,
        data=data,
        config=submission_config,
        output_path=submission_path,
        device=device,
    )

    assert list(submission.columns) == ["filament_id", "segmentation_rle"]
    assert submission["filament_id"].is_unique
    assert not submission.isna().any().any()
    assert len(submission) > 0

    test_paths = sorted(
        p for p in Path(data.test_images).iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    )
    test_stems = {p.stem for p in test_paths}
    output_stems = {
        str(fid).rsplit("_", 1)[0]
        for fid in submission["filament_id"].tolist()
    }
    assert output_stems <= test_stems, "Submission contains unknown observation IDs."

    print("SUBMISSION CREATED:", submission_path)
    print("Test observations:", len(test_paths))
    print("Submission rows:", len(submission))
    print("Predicted instances per test observation:", round(len(submission) / max(1, len(test_paths)), 2))
    display(submission.head(10))
else:
    print("SMOKE MODE: test submission disabled intentionally.")


In [ ]:
# ============================================================
# CELL 14 — REPRODUCIBILITY SUMMARY
# ============================================================
summary = {
    "run_tag": RUN_TAG,
    "repo_branch": REPO_BRANCH,
    "repo_commit": REPO_COMMIT,
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision_available": TORCHVISION_OK,
    "torchvision_error": TORCHVISION_ERROR,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "run_mode": RUN_MODE,
    "best_epoch": best_epoch,
    "best_quick_global_pq": best_metric,
    "independent_holdout_metrics": holdout_metrics,
    "holdout_eval_seconds": holdout_seconds,
    "num_train_annotation_views": len(train_ids),
    "num_val_annotation_views": len(val_ids),
    "num_train_physical_images": len(physical_train),
    "num_val_physical_images": len(physical_val),
    "final_checkpoint": str(final_model_path),
    "submission_path": str(WORKING / "submission.csv") if RUN_MODE == "competition" else None,
}

summary_path = WORKING / f"run_summary_{RUN_TAG}.json"
summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print(json.dumps(summary, indent=2, default=str))
print("\nArtifacts in /kaggle/working can be downloaded from the notebook Output panel.")
